# Debugging Queries

## Import Libraries

In [1]:
from pathlib import Path
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain.schema import Document
from langchain_openai import OpenAIEmbeddings


import os
import pandas as pd
import numpy as np
import chromadb


## Initialize DB

In [12]:
embedding_function = OpenAIEmbeddings()
vector_db = Chroma(persist_directory="../db/dunder_bot", embedding_function=embedding_function,collection_metadata={"hnsw:M": 1024,"hnsw:ef": 64})

In [13]:
## lets verify the db connection
vector_db._collection.count()

14998

In [14]:
## Retrieve using similarity search
query = "When did Michael say 'I love you'"
results = vector_db.similarity_search(query=query, k=4)

for doc in results:
    print(doc.page_content)
    print(doc.metadata)

in love on Valentine's Day. Holly: Two people in love? Michael: I love you. Holly: Wait, wait, wait, what do you mean you love me? We've only been dating for a week. Do you mean you love me like, 'oh, hey, there's Holly. I love that girl.' Or you do you mean you love me like you love me-love me? Michael: I love you-love you.
{'directed_by': 'Greg Daniels', 'episode': 15, 'episode_description': "It's Valentine's Day, and the office is fed up with Michael and Holly's PDA, Andy helps Erin solve Gabe's riddles to find her gift, and Jim and Pam get drunk and try to find a place in the office to have sex.", 'id': 's7_e15_scene23', 'rating': 8.4, 'scene': 23, 'season': 7, 'speakers': 'Holly,Michael', 'written_by': 'Robert Padnick'}
I'm too shy to tell you that I love you.' Michael: Pam.  Pam, you gave me your word. Ryan: [kissing Kelly against her desk] You did that for me? Kelly: Mmhmm. Ryan: Are you happy you did? Toby: Hey guys that's really inappropriate. Ryan: [kisses for a little longer

### Debugging the following query from production app
```json
{
    'user_query': 'Jim first line Season 1 Episode 1', 
    'number_of_results': 3, 
    'filter': {
        '$and': [
            {'season': {'$eq': 1}}, 
            {'episode': {'$eq': 1}}, 
            {'speakers': {'$in': ['Jim']}}
        ]
    }
}
```

In [ ]:
query = 'grasshoper'
filter = {
        '$and': [
            {'season': {'$eq': 1}}, 
            {'episode': {'$eq': 1}}, 
            # {'speakers': {'$in': ['Jim']}}
        ]
    }

matching_docs = vector_db._collection.get(where=filter)
print(matching_docs["ids"])
# k = min(user_provided_k, matching_docs)
# results = vector_db.similarity_search(query=query, k=1, filter=filter)

# print(results)
# for doc in results:
#     print(doc.page_content)
#     print(doc.metadata)



['s1_e1_scene1_chunk0', 's1_e1_scene1_chunk1', 's1_e1_scene1_chunk2', 's1_e1_scene1_chunk3', 's1_e1_scene1_chunk4', 's1_e1_scene2_chunk5', 's1_e1_scene3_chunk6', 's1_e1_scene3_chunk7', 's1_e1_scene3_chunk8', 's1_e1_scene4_chunk9', 's1_e1_scene5_chunk10', 's1_e1_scene5_chunk11', 's1_e1_scene6_chunk12', 's1_e1_scene6_chunk13', 's1_e1_scene7_chunk14', 's1_e1_scene7_chunk15', 's1_e1_scene8_chunk16', 's1_e1_scene8_chunk17', 's1_e1_scene9_chunk18', 's1_e1_scene9_chunk19', 's1_e1_scene10_chunk20', 's1_e1_scene11_chunk21', 's1_e1_scene12_chunk22', 's1_e1_scene12_chunk23', 's1_e1_scene12_chunk24', 's1_e1_scene13_chunk25', 's1_e1_scene14_chunk26', 's1_e1_scene14_chunk27', 's1_e1_scene15_chunk28', 's1_e1_scene15_chunk29', 's1_e1_scene16_chunk30', 's1_e1_scene16_chunk31', 's1_e1_scene17_chunk32', 's1_e1_scene17_chunk33', 's1_e1_scene18_chunk34', 's1_e1_scene19_chunk35', 's1_e1_scene19_chunk36', 's1_e1_scene20_chunk37', 's1_e1_scene20_chunk38', 's1_e1_scene20_chunk39', 's1_e1_scene21_chunk40', 's1_

In [38]:
matching_docs = vector_db._collection.get(ids="s1_e1_scene1_chunk0")
matching_docs

{'ids': ['s1_e1_scene1_chunk0'],
 'embeddings': None,
 'documents': ['--- Episode Start ---\nThe premiere episode introduces the boss and staff of the Dunder-Mifflin Paper Company in Scranton, Pennsylvania in a documentary about the workplace.'],
 'uris': None,
 'data': None,
 'metadatas': [{'directed_by': 'Ken Kwapis',
   'episode': 1,
   'episode_description': 'The premiere episode introduces the boss and staff of the Dunder-Mifflin Paper Company in Scranton, Pennsylvania in a documentary about the workplace.',
   'id': 's1_e1_scene1',
   'rating': 7.5,
   'scene': 1,
   'season': 1,
   'speakers': 'Michael,Jim',
   'written_by': 'Ricky Gervais,Stephen Merchant,Greg Daniels'}],
 'included': [<IncludeEnum.documents: 'documents'>,
  <IncludeEnum.metadatas: 'metadatas'>]}